# Random Forest - Classificação

Os dados serão divididos em treinamento (80%) e teste (20%). 

O modelo será treinado com n_estimators=100 e profundidade limitada para evitar overfitting, classificando a qualidade da água (ex.: adequada/inadequada, conforme Resolução 
CONAMA 357/2005, Brasil, 2012).  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pcj.utils import BASE_DIR, dados

In [ ]:
# Dados para testes

dados_limpos, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")

X = dados_limpos.copy()

In [ ]:
# Verificações

# print(dados_limpos['Data'].dtype, end='')
# print(dados_limpos['data_normalizada'].dtype)
# print(dados_limpos.columns)
# print(variaveis)
print(dados_limpos[variaveis])

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 01 - Minha Tentativa

In [ ]:
from sklearn.ensemble import RandomForestClassifier # Documentação: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
from sklearn.datasets import make_classification    # Lendo a documentação (https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html) 
                                                    # desse método, acho que ele só serve para gerar um problema aleatório de classificação.

# Estabelece as regras para o funcionamento do Classificador
clf = RandomForestClassifier(n_estimators=100,  # n_estimators=100 é default, mas como está nas instruções do relatório decidi evidenciar.
                             
                             bootstrap=True,    # bootstrap=True também é default, mas acho que é importante ele ser True para determinar a divisão de dados de treinamento e teste. Não sei ao certo.

                             max_samples=0.8,   # Pelo meu entendimento, isso deveria ser a quantidade de dados utilizados no treinamento, portanto, 80% do número de linhas (0.8 * shape[0]).
                                                                    # InvalidParameterError: The 'max_samples' parameter of RandomForestClassifier must be None, a float in the range (0.0, 1.0] or an int in the range [1, inf). Got 460.8 instead.
                                                                    # Acho que o certo então é só colocar 0.8 --> Funcionou

                            #  n_classes=2      # Acho que como nós queremos classificar a qualidade da água em "adequada" ou "inadequada", n_classes=2. 
                                                # Esquece: RandomForestClassifier.__init__() got an unexpected keyword argument 'n_classes'  
                                                                                     
                            #  n_features=      # Não entendo, mas parece importante
                             ) 

# Faz o fit do classificador, necessário para o funcionamento de estimators (NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.)

clf.fit(dados_limpos)   # Preciso de um argumento X e um y, mas não sei ao certo o que seria qual.

In [ ]:
# clf.apply(dados_limpos)

clf.decision_path(dados_limpos)

In [ ]:
dados_limpos.shape[0]*.8

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 02 - Seguinto Tutorial

Link para o tutorial: https://hub.asimov.academy/blog/random-forest/

Observações: 
1) Limitações parecidas com o KNN. O tutorial instrui para um Classifier, mas, como temos dados contínuos, é preciso usar um Regressor.
2) Com isso, os métodos de cálculo de acurácia não se aplicam, então utilizamos métricas de avaliação.

In [ ]:
# Carregar e preparar dados
var_alvo = 'pH' # Estou selecionando pH como a variável alvo
x = X.drop(var_alvo, axis=1) 
y = X[var_alvo]

In [ ]:
# Divisão de treino e teste
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                    test_size=0.3,  # Estou copiando do tutorial
                                                    random_state=42
                                                    )

In [ ]:
# Criando o modelo
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100,    # Copiando do tutorial
                               random_state=42)

In [ ]:
# Treinando o modelo
model.fit(x_train, y_train)

In [ ]:
# Previsões
y_pred = model.predict(x_test)

In [ ]:
# Avaliação do Desempenho
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

print("RMSE:", root_mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("r²:", r2_score(y_test, y_pred))

![green-divider](https://user-images.githubusercontent.com/7065401/52071924-c003ad80-2562-11e9-8297-1c6595f8a7ff.png)

### 03 - Seguindo outro tutorial

Link para o tutorial: https://medium.com/cinthiabpessanha/random-forest-como-funciona-um-dos-algoritmos-mais-populares-de-ml-cc1b8a58b3b4

Observações: 
1) Mesmas limitações dos dados contínuos;
2) O tutorial não é muito bem detalhado, então vou reutilizar o código do tutorial anterior;
3) Achei interessante a utilização da função de validação cruzada, vou tentar replicar

In [ ]:
# Carregar e preparar dados
var_alvo = 'pH' # Estou selecionando pH como a variável alvo
x = X.drop(var_alvo, axis=1) 
y = X[var_alvo]

In [ ]:
# Divisão de treino e teste
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, 
                                                    test_size=0.3,  # Estou copiando do tutorial
                                                    random_state=42
                                                    )

In [ ]:
# Criando o modelo
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=100,    # Copiando do tutorial
                               random_state=42)

In [ ]:
# Treinando o modelo
model_rf = model.fit(x_train, y_train)

In [ ]:
# Previsões
y_pred = model_rf.predict(x_test)

In [ ]:
# Validação Cruzada
from sklearn.model_selection import cross_val_score

print(cross_val_score(model_rf, x_test, y_test, cv=10))

# O tutorial também faz o accuracy score, que eu deveria adequar para
# as métricas, mas acho que acabaria dando na mesma.